# Train dịch Ê Đê – Việt trên Google Colab

**Chuẩn bị (một lần):**
1. Tạo thư mục `Hriam` trong Google Drive và tải file `dataset_colab.zip` vào đó (file nằm ở `data/dataset_colab.zip` trên máy).
2. Trong Colab chọn **Runtime → Change runtime type → GPU**.
3. Chọn **Runtime → Run all** và cho phép truy cập Drive.

**Khi phiên bị ngắt:** mở lại notebook và **Run all**, notebook sẽ chạy tiếp từ checkpoint trên Drive.

**Kết quả** nằm trong `Drive/Hriam/runs/<hướng>/`: `best/`, `train.log`, `test_results.json`, `predictions_*.jsonl`.

Mô hình NLLB có giấy phép CC-BY-NC, chỉ dùng phi thương mại. NLLB không có tiếng Ê Đê nên mượn mã `ace_Latn` (tiếng Aceh, cùng nhóm Chăm).

## 1. Cài thư viện

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("google.colab"):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "transformers==5.22.1", "sacrebleu==2.6.0",
                    "sentencepiece", "protobuf"], check=True)

## 2. Cấu hình

Thường chỉ cần đổi `DIRECTION`. Muốn thử nhanh trước thì đặt `LIMIT = 200`, nhưng nhớ xoá thư mục `runs/` tương ứng trước khi train thật.

In [ ]:
DIRECTION = "ede2vi"   # "ede2vi" hoặc "vi2ede"
EPOCHS = 4
LEARNING_RATE = 5e-5
BEAMS = 4
SAVE_EVERY = 500       # lưu checkpoint lên Drive sau mỗi N bước
LIMIT = None           # None = toàn bộ dữ liệu

MODEL_NAME = "facebook/nllb-200-distilled-600M"
LANG_CODES = {"ede": "ace_Latn", "vi": "vie_Latn"}
DIRECTIONS = {"ede2vi": ("ede", "vi"), "vi2ede": ("vi", "ede")}
# Câu càng dài thì batch càng nhỏ. Nếu log báo tràn bộ nhớ nhiều lần, hãy giảm các số này.
BATCH_BY_LENGTH = {32: 32, 64: 16, 96: 7, 128: 4}
MAX_LENGTH = max(BATCH_BY_LENGTH)
SEED = 42

## 3. Kết nối Drive và giải nén dữ liệu

In [ ]:
import importlib.util, json, os, random, shutil, sys, time, zipfile
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/Hriam")
    DATA_DIR = Path("/content/dataset")
else:
    BASE = Path(".")
    DATA_DIR = Path("dataset_unzipped")

archive = BASE / "dataset_colab.zip"
if not archive.exists():
    raise FileNotFoundError(f"Không thấy {archive}. Hãy tải dataset_colab.zip vào thư mục Hriam trên Drive.")
with zipfile.ZipFile(archive) as zf:
    zf.extractall(DATA_DIR)

RUN_DIR = BASE / "runs" / DIRECTION
RUN_DIR.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(BASE).free / 1e9
print(f"Drive/ổ đĩa còn trống {free_gb:.1f} GB")
if free_gb < 8 and not (RUN_DIR / "last").exists():
    print("CẢNH BÁO: nên có ít nhất 8 GB trống để lưu checkpoint.")

## 4. Nạp dữ liệu và mô hình

In [ ]:
import sacrebleu
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
else:
    print("Thiết bị:", DEVICE)
if IN_COLAB and DEVICE != "cuda":
    raise RuntimeError("Chưa bật GPU. Vào Runtime → Change runtime type → GPU rồi chạy lại.")

SRC, TGT = DIRECTIONS[DIRECTION]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=LANG_CODES[SRC], tgt_lang=LANG_CODES[TGT])
target_id = tokenizer.convert_tokens_to_ids(LANG_CODES[TGT])

def log(message):
    print(message, flush=True)
    with (RUN_DIR / "train.log").open("a", encoding="utf-8") as out:
        out.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} {message}\n")

def load_split(name):
    with (DATA_DIR / f"{name}.jsonl").open(encoding="utf-8") as source:
        rows = [json.loads(line) for line in source]
    if LIMIT and LIMIT < len(rows):
        rows = random.Random(SEED).sample(rows, LIMIT)
    sources = tokenizer([r[SRC] for r in rows])["input_ids"]
    targets = tokenizer(text_target=[r[TGT] for r in rows])["input_ids"]
    for row, s, t in zip(rows, sources, targets):
        row["length"] = max(len(s), len(t))
    return rows

train_rows = load_split("train")
total = len(train_rows)
train_rows = [r for r in train_rows if r["length"] <= MAX_LENGTH]
log(f"train: dùng {len(train_rows)}/{total} cặp, bỏ {total - len(train_rows)} cặp dài hơn {MAX_LENGTH} token.")
dev_sets = {name: load_split(name) for name in ("dev_bible", "dev_dict")}
test_sets = {name: load_split(name) for name in ("test_bible", "test_dict")}
for name, rows in {**dev_sets, **test_sets}.items():
    log(f"{name}: {len(rows)} cặp, {sum(r['length'] > MAX_LENGTH for r in rows)} cặp có câu bị cắt khi đánh giá.")

## 5. Các hàm huấn luyện và đánh giá

Checkpoint trên Drive chỉ gồm trọng số mô hình và vị trí đang chạy, không gồm trạng thái optimizer (khoảng 5 GB), để không tràn dung lượng 15 GB. Khi chạy tiếp, optimizer khởi tạo lại, ảnh hưởng thường không đáng kể.

In [ ]:
def bucket_length(length):
    return next((size for size in sorted(BATCH_BY_LENGTH) if length <= size), MAX_LENGTH)

def make_batches(rows, epoch):
    rng = random.Random(SEED + epoch)
    groups = {}
    for row in rows:
        groups.setdefault(bucket_length(row["length"]), []).append(row)
    batches = []
    for size in sorted(groups):
        group = groups[size]
        rng.shuffle(group)
        step = BATCH_BY_LENGTH[size]
        batches += [(size, group[i:i + step]) for i in range(0, len(group), step)]
    rng.shuffle(batches)
    return batches

@torch.no_grad()
def translate(rows):
    model.eval()
    groups = {}
    for i, row in enumerate(rows):
        groups.setdefault(bucket_length(row["length"]), []).append(i)
    output = [""] * len(rows)
    for size, indices in sorted(groups.items()):
        step = max(1, 2 * BATCH_BY_LENGTH[size] // BEAMS)
        for start in range(0, len(indices), step):
            part = indices[start:start + step]
            batch = tokenizer([rows[i][SRC] for i in part], return_tensors="pt", padding="max_length",
                              truncation=True, max_length=size).to(DEVICE)
            ids = model.generate(**batch, forced_bos_token_id=target_id, num_beams=BEAMS,
                                 max_new_tokens=MAX_LENGTH, no_repeat_ngram_size=3)
            for i, text in zip(part, tokenizer.batch_decode(ids, skip_special_tokens=True)):
                output[i] = text
    return output

def evaluate(sets, save_predictions=False):
    scores = {}
    for name, rows in sets.items():
        start = time.time()
        hyps = translate(rows)
        refs = [r[TGT] for r in rows]
        scores[name] = {"bleu": round(sacrebleu.corpus_bleu(hyps, [refs]).score, 2),
                        "chrf": round(sacrebleu.corpus_chrf(hyps, [refs]).score, 2),
                        "n": len(rows), "seconds": round(time.time() - start)}
        if save_predictions:
            with (RUN_DIR / f"predictions_{name}.jsonl").open("w", encoding="utf-8") as out:
                for row, hyp in zip(rows, hyps):
                    out.write(json.dumps({"source": row[SRC], "reference": row[TGT], "output": hyp,
                                          "ref": row.get("ref")}, ensure_ascii=False) + "\n")
            for row, hyp in list(zip(rows, hyps))[:5]:
                log(f"[{name}]\n  Nguồn: {row[SRC]}\n  Đích : {row[TGT]}\n  Dịch : {hyp}")
    return scores

def save(path, with_state=False):
    # Ghi vào thư mục tạm rồi đổi tên để phiên bị ngắt không làm hỏng bản đang có.
    temp = path.with_name(path.name + ".tmp")
    shutil.rmtree(temp, ignore_errors=True)
    temp.mkdir(parents=True)
    model.save_pretrained(temp)
    tokenizer.save_pretrained(temp)
    if with_state:
        (temp / "progress.json").write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")
    shutil.rmtree(path, ignore_errors=True)
    temp.rename(path)

## 6. Huấn luyện

Ô này chạy lâu nhất. Log được ghi cả vào `train.log` trên Drive.

In [ ]:
last_dir, best_dir = RUN_DIR / "last", RUN_DIR / "best"
state = {"epoch": 0, "step": 0, "best": -1.0, "history": [], "before": None, "skipped": 0}
source = MODEL_NAME
if (last_dir / "progress.json").exists():
    state = json.loads((last_dir / "progress.json").read_text(encoding="utf-8"))
    source = last_dir
    log(f"Chạy tiếp từ epoch {state['epoch'] + 1}, bước {state['step']}.")
model = AutoModelForSeq2SeqLM.from_pretrained(source).to(DEVICE)
model.generation_config.max_length = None

steps_per_epoch = len(make_batches(train_rows, 0))
total_steps = EPOCHS * steps_per_epoch
warmup = max(1, int(0.05 * total_steps))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: min((s + 1) / warmup, max(0.0, (total_steps - s) / max(1, total_steps - warmup))))
done_steps = state["epoch"] * steps_per_epoch + state["step"]
for _ in range(done_steps):
    scheduler.step()
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
log(f"Hướng {DIRECTION}, {EPOCHS} epoch, mỗi epoch {steps_per_epoch} bước.")

if state["before"] is None:
    state["before"] = evaluate(dev_sets)
    log(f"Dev trước khi train: {state['before']}")

for epoch in range(state["epoch"], EPOCHS):
    model.train()
    batches = make_batches(train_rows, epoch)
    first = state["step"]
    started, pairs, running, counted = time.time(), 0, 0.0, 0
    for step, (size, batch) in enumerate(batches[first:], first + 1):
        encoded = tokenizer([r[SRC] for r in batch], text_target=[r[TGT] for r in batch], return_tensors="pt",
                            padding="max_length", truncation=True, max_length=size)
        encoded["labels"][encoded["labels"] == tokenizer.pad_token_id] = -100
        encoded = encoded.to(DEVICE)
        try:
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                loss = model(**encoded).loss
            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss không hợp lệ ở epoch {epoch + 1}, bước {step}: {loss.item()}")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            running += loss.item()
            counted += 1
            pairs += len(batch)
        except torch.cuda.OutOfMemoryError:
            state["skipped"] += 1
            log(f"Tràn bộ nhớ ở bước {step} (batch {len(batch)} × {size}), bỏ qua batch này. "
                f"Tổng đã bỏ: {state['skipped']}. Nếu số này tăng nhiều, hãy giảm BATCH_BY_LENGTH.")
            torch.cuda.empty_cache()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        state["step"] = step
        if counted and (step - first) % 100 == 0:
            speed = pairs / (time.time() - started)
            remaining = len(train_rows) * (EPOCHS - epoch) - sum(len(b) for _, b in batches[:step])
            log(f"epoch {epoch + 1}/{EPOCHS} bước {step}/{len(batches)} loss {running / counted:.3f} "
                f"{speed:.1f} cặp/giây, còn khoảng {remaining / speed / 3600:.1f} giờ (chưa tính đánh giá)")
        if step < len(batches) and step % SAVE_EVERY == 0:
            save(last_dir, with_state=True)
    scores = evaluate(dev_sets)
    mean_chrf = sum(s["chrf"] for s in scores.values()) / len(scores)
    state["epoch"], state["step"] = epoch + 1, 0
    state["history"].append({"epoch": epoch + 1, "loss": running / max(1, counted), "dev": scores})
    log(f"Dev sau epoch {epoch + 1}: {scores}")
    if mean_chrf > state["best"]:
        state["best"] = mean_chrf
        save(best_dir)
        log(f"Lưu bản tốt nhất, chrF trung bình {mean_chrf:.2f}.")
    save(last_dir, with_state=True)

## 7. Chấm trên tập test

Tập test chỉ được dùng ở đây, sau khi đã chọn mô hình bằng tập dev.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(best_dir).to(DEVICE)
model.generation_config.max_length = None
results = evaluate(test_sets, save_predictions=True)
(RUN_DIR / "test_results.json").write_text(json.dumps(
    {"test": results, "before_training_dev": state["before"], "history": state["history"]},
    ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
log(f"TEST: {results}")
log(f"Xong. Mô hình tốt nhất ở {best_dir}")

## 8. Thử dịch câu của bạn

In [ ]:
for text in ["Kâo nao sang čơ", "Ih suaih asei mlei mơ̆?"]:
    row = {SRC: text, "length": len(tokenizer(text)["input_ids"])}
    print(text, "→", translate([row])[0])